# Paso 1. Extracción de datos

Accediendo a la página [Basketball Reference](https://www.basketball-reference.com) para obtener los datos de jugadores, equipos y votaciones de cada año.

In [1]:
import pandas as pd
import numpy as np
import time as time
import re

In [2]:
def resolve_team_abbr(team_name, year):
    """
    Mapeo dinámico de nombres de equipos a siglas oficiales de Basketball-Reference,
    resolviendo conflictos históricos de Charlotte y New Orleans.
    """
    team_name = re.sub(r'[\*\(\)]', '', team_name).strip()

    # Los Hornets tienen varias siglas a lo largos de los años dependiendo de la ubicacion del club
    if 'Charlotte Hornets' in team_name:
        return 'CHO' if year >= 2015 else 'CHA'
    if 'Charlotte Bobcats' in team_name:
        return 'CHA'

    # Caso similar para los Pelicans
    if 'New Orleans' in team_name:
        if 2006 <= year <= 2007: return 'NOK'
        return 'NOH' if year <= 2013 else 'NOP'

    standard_map = {
        'Atlanta Hawks': 'ATL', 'Boston Celtics': 'BOS', 'Brooklyn Nets': 'BRK',
        'Chicago Bulls': 'CHI', 'Cleveland Cavaliers': 'CLE', 'Dallas Mavericks': 'DAL',
        'Denver Nuggets': 'DEN', 'Detroit Pistons': 'DET', 'Golden State Warriors': 'GSW',
        'Houston Rockets': 'HOU', 'Indiana Pacers': 'IND', 'Los Angeles Clippers': 'LAC',
        'Los Angeles Lakers': 'LAL', 'Memphis Grizzlies': 'MEM', 'Miami Heat': 'MIA',
        'Milwaukee Bucks': 'MIL', 'Minnesota Timberwolves': 'MIN', 'New York Knicks': 'NYK',
        'Oklahoma City Thunder': 'OKC', 'Orlando Magic': 'ORL', 'Philadelphia 76ers': 'PHI',
        'Phoenix Suns': 'PHO', 'Portland Trail Blazers': 'POR', 'Sacramento Kings': 'SAC',
        'San Antonio Spurs': 'SAS', 'Toronto Raptors': 'TOR', 'Utah Jazz': 'UTA',
        'Washington Wizards': 'WAS', 'New Jersey Nets': 'NJN', 'Seattle SuperSonics': 'SEA',
        'Vancouver Grizzlies': 'VAN', 'San Diego Clippers': 'SDC', 'Kansas City Kings': 'KCK',
        'Washington Bullets': 'WSB'
    }
    return standard_map.get(team_name, None)

def get_nba_data(year):
    """
    Extrae y limpia datos de stats, standings y votaciones.
    """
    url_stats = f"https://www.basketball-reference.com/leagues/NBA_{year}_advanced.html"
    url_standings = f"https://www.basketball-reference.com/leagues/NBA_{year}_standings.html"
    url_awards = f"https://www.basketball-reference.com/awards/awards_{year}.html"

    df_stats = pd.read_html(url_stats)[0]
    df_stats = df_stats[df_stats['Player'] != 'Player'].copy()
    df_stats['Player'] = df_stats['Player'].str.replace('*', '', regex=False)
    df_stats = df_stats.drop_duplicates(subset=['Player'], keep='first')

    try:
        tables = pd.read_html(url_standings)
        if len(tables) >= 2:
            east, west = tables[0], tables[1]
            cols = ['Team', 'W', 'L', 'W/L%', 'GB', 'PS/G', 'PA/G', 'SRS']
            east.columns, west.columns = cols, cols
            df_standings = pd.concat([east, west])
        else:
            df_standings = tables[0]
            df_standings.columns = ['Team', 'W', 'L', 'W/L%', 'GB', 'PS/G', 'PA/G', 'SRS']
    except:
        df_standings = pd.DataFrame(columns=['Team', 'W/L%'])

    df_standings = df_standings[~df_standings['Team'].str.contains('Division|Conference', na=False, case=False)]

    try:
        df_mvp = pd.read_html(url_awards)[0]
        df_mvp.columns = df_mvp.columns.get_level_values(-1)
        df_mvp = df_mvp[['Player', 'Share']]
    except:
        df_mvp = pd.DataFrame(columns=['Player', 'Share'])

    return df_stats, df_standings, df_mvp

def build_definitive_dataset(years):
    all_seasons = []

    for year in years:
        print(f"Procesando temporada {year}...")
        try:
            stats, standings, awards = get_nba_data(year)

            standings['W'] = pd.to_numeric(standings['W'], errors='coerce')
            standings['L'] = pd.to_numeric(standings['L'], errors='coerce')

            total_season_games = (standings['W'] + standings['L']).max()

            if pd.isna(total_season_games) or total_season_games == 0:
                total_season_games = 82

            # Calculamos margen de partidos para filtrar los jugadores
            games_threshold = round(0.67 * total_season_games)

            stats['PER'] = pd.to_numeric(stats['PER'], errors='coerce')
            stats['G'] = pd.to_numeric(stats['G'], errors='coerce')
            standings['W/L%'] = pd.to_numeric(standings['W/L%'], errors='coerce')

            df = pd.merge(stats, awards, on='Player', how='left').fillna({'Share': 0})

            standings['Team_Abbr'] = standings['Team'].apply(lambda x: resolve_team_abbr(x, year))

            # Normalizamos de la misma manera las siglas de los equipos como antes
            df['Team'] = df['Team'].replace({'CHH': 'CHA'})
            if year >= 2015:
                df['Team'] = df['Team'].replace({'CHA': 'CHO'})

            df = pd.merge(df, standings[['Team_Abbr', 'W/L%']], left_on='Team', right_on='Team_Abbr', how='left')

            candidates = df[
                (df['G'] >= games_threshold) &               # Mínimo de partidos (67% del total de la temporada)
                (df['PER'] >= 18.0)                           # Eficiencia mínima (Marcado de manera fija)
            ].copy()

            candidates['Season'] = year
            all_seasons.append(candidates)

            print(f"  -> {len(candidates)} candidatos filtrados (minimo de partidos: {games_threshold}, total partidos temporada: {total_season_games}).")
            time.sleep(10) # Necesitamos margen para que no de problemas de acceso a la pagina

        except Exception as e:
            print(f"Error en temporada {year}: {e}")

    final_df = pd.concat(all_seasons) if all_seasons else pd.DataFrame()
    return final_df

In [3]:
years_list = range(1980, 2026) # El TFM se marcará entre 1980 y el último año que existen datos de votaciones.
final_df = build_definitive_dataset(years_list)
final_df

Procesando temporada 1980...
  -> 34 candidatos filtrados (minimo de partidos: 55, total partidos temporada: 82).
Procesando temporada 1981...
  -> 36 candidatos filtrados (minimo de partidos: 55, total partidos temporada: 82).
Procesando temporada 1982...
  -> 33 candidatos filtrados (minimo de partidos: 55, total partidos temporada: 82).
Procesando temporada 1983...
  -> 39 candidatos filtrados (minimo de partidos: 55, total partidos temporada: 82).
Procesando temporada 1984...
  -> 37 candidatos filtrados (minimo de partidos: 55, total partidos temporada: 82).
Procesando temporada 1985...
  -> 36 candidatos filtrados (minimo de partidos: 55, total partidos temporada: 82).
Procesando temporada 1986...
  -> 35 candidatos filtrados (minimo de partidos: 55, total partidos temporada: 82).
Procesando temporada 1987...
  -> 34 candidatos filtrados (minimo de partidos: 55, total partidos temporada: 82).
Procesando temporada 1988...
  -> 34 candidatos filtrados (minimo de partidos: 55, total

,Rk,Player,Age,Team,Pos,G,GS,MP,PER,TS%,...,WS/48,OBPM,DBPM,BPM,VORP,Awards,Share,Team_Abbr,W/L%,Season
3,4.0,Kareem Abdul-Jabbar,32.0,LAL,C,82.0,NaN,3143.0,25.3,0.639,...,0.227,4.8,2.4,7.2,7.3,"MVP-1,AS,NBA1,DEF1",0.665,LAL,0.732,1980
4,5.0,Moses Malone,24.0,HOU,C,82.0,82.0,3140.0,24.1,0.560,...,0.183,4.5,-2.4,2.1,3.2,"MVP-9,AS,NBA2",0.005,HOU,0.500,1980
9,10.0,Gus Williams,26.0,SEA,PG,82.0,82.0,2969.0,20.6,0.529,...,0.187,3.1,1.6,4.7,5.0,"MVP-8,NBA2",0.007,SEA,0.683,1980
10,11.0,Larry Bird,23.0,BOS,PF,82.0,82.0,2955.0,20.5,0.538,...,0.182,3.0,1.5,4.5,4.8,"MVP-4,ROY-1,AS,NBA1",0.068,BOS,0.744,1980
12,13.0,Dan Issel,31.0,DEN,C,82.0,82.0,2938.0,22.2,0.571,...,0.188,4.1,-0.8,3.3,3.9,NaN,0.000,DEN,0.366,1980
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
230,231.0,Daniel Gafford,26.0,DAL,C,57.0,31.0,1226.0,24.7,0.716,...,0.232,2.7,1.1,3.8,1.8,NaN,0.000,DAL,0.476,2025
274,275.0,Trayce Jackson-Davis,24.0,GSW,C,62.0,37.0,967.0,18.5,0.587,...,0.167,0.3,0.9,1.2,0.8,NaN,0.000,GSW,0.585,2025
276,277.0,Jalen Smith,24.0,CHI,C,64.0,2.0,962.0,18.7,0.586,...,0.146,1.5,-0.6,0.9,0.7,NaN,0.000,CHI,0.476,2025
320,321.0,Jay Huff,26.0,MEM,C,64.0,2.0,748.0,18.2,0.666,...,0.162,2.5,1.0,3.5,1.1,NaN,0.000,MEM,0.585,2025


In [4]:
final_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 2082 entries, 3 to 329
Data columns (total 33 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   Rk         2082 non-null   float64
 1   Player     2082 non-null   object 
 2   Age        2082 non-null   float64
 3   Team       2082 non-null   object 
 4   Pos        2082 non-null   object 
 5   G          2082 non-null   float64
 6   GS         2039 non-null   float64
 7   MP         2082 non-null   float64
 8   PER        2082 non-null   float64
 9   TS%        2082 non-null   float64
 10  3PAr       2082 non-null   float64
 11  FTr        2082 non-null   float64
 12  ORB%       2082 non-null   float64
 13  DRB%       2082 non-null   float64
 14  TRB%       2082 non-null   float64
 15  AST%       2082 non-null   float64
 16  STL%       2082 non-null   float64
 17  BLK%       2082 non-null   float64
 18  TOV%       2082 non-null   float64
 19  USG%       2082 non-null   float64
 20  OWS        208

In [5]:
final_df.to_csv('datos_1980-2025.csv', index=False) # Almacenamos datos brutos